# Feature Engineering

This notebook demonstrates the domain-driven features engineered in [`src/features/build_features.py`](../src/features/build_features.py) and inspects the resulting preprocessing pipeline (imputation → scaling → one-hot encoding) that feeds the modeling stage.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.features.build_features import engineer_features, build_preprocessor

df = pd.read_csv('../data/processed/churn_clean.csv')
df_feat = engineer_features(df)
print(f'Before: {df.shape[1]} columns | After: {df_feat.shape[1]} columns')

Before: 20 columns | After: 27 columns


## New engineered features

| Feature | Description |
|---|---|
| `tenure_bucket` | Categorical tenure cohort (New / Established / Loyal / VIP) |
| `num_active_services` | Count of subscribed add-on services (0-8) |
| `has_internet` | Binary flag for internet subscription |
| `avg_monthly_spend` | `TotalCharges / tenure` — smoothed historical billing |
| `charge_per_service` | `MonthlyCharges / (num_active_services + 1)` — pricing efficiency |
| `is_month_to_month` | Binary flag — the single strongest churn predictor |
| `contract_risk_score` | Ordinal encoding of contract commitment (0-2) |


In [2]:
new_cols = ['tenure_bucket', 'num_active_services', 'has_internet',
            'avg_monthly_spend', 'charge_per_service', 'is_month_to_month', 'contract_risk_score']
df_feat[new_cols + ['Churn']].head(10)

,tenure_bucket,num_active_services,has_internet,avg_monthly_spend,charge_per_service,is_month_to_month,contract_risk_score,Churn
0,New(0-12mo),1,1,29.850000,14.925000,1,0,0
1,Loyal(2-4yr),3,1,55.573529,14.237500,0,1,0
2,New(0-12mo),3,1,54.075000,13.462500,1,0,1
3,Loyal(2-4yr),3,1,40.905556,10.575000,0,1,0
4,New(0-12mo),1,1,75.825000,35.350000,1,0,1
5,New(0-12mo),5,1,102.562500,16.608333,1,0,1
6,Established(1-2yr),4,1,88.609091,17.820000,1,0,0
7,New(0-12mo),1,1,30.190000,14.875000,1,0,0
8,Loyal(2-4yr),6,1,108.787500,14.971429,1,0,1
9,VIP(4yr+),3,1,56.257258,14.037500,0,1,0


## Engineered feature relationship with churn

Sanity-check that the engineered features carry real signal before they go into the model.

In [3]:
df_feat.groupby('tenure_bucket', observed=True)['Churn'].mean().sort_values(ascending=False)

tenure_bucket
New(0-12mo)           0.473660
Established(1-2yr)    0.287109
Loyal(2-4yr)          0.203890
VIP(4yr+)             0.095132
Name: Churn, dtype: float64

In [4]:
df_feat.groupby('num_active_services')['Churn'].mean()

num_active_services
0    0.437500
1    0.206671
2    0.328283
3    0.364767
4    0.313449
5    0.255507
6    0.224852
7    0.124051
8    0.052885
Name: Churn, dtype: float64

Churn risk drops sharply as customers accumulate more add-on services — a clean, monotonic relationship the tree-based models can exploit directly.

## Preprocessing pipeline

Numeric features are standard-scaled; categorical features are one-hot encoded with `handle_unknown='ignore'` so the pipeline stays robust to unseen categories at inference time. The full `ColumnTransformer` is fit once on the training split and reused identically for cross-validation, testing, the FastAPI service, and the Streamlit dashboard — eliminating train/serve skew.

In [5]:
preprocessor, numeric_cols, categorical_cols = build_preprocessor(df_feat)
print(f'{len(numeric_cols)} numeric features:', numeric_cols)
print(f'{len(categorical_cols)} categorical features:', categorical_cols)

2026-08-15 21:55:01,943 | INFO | Numeric features (9): ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_active_services', 'has_internet', 'avg_monthly_spend', 'charge_per_service', 'is_month_to_month', 'contract_risk_score']


2026-08-15 21:55:01,943 | INFO | Categorical features (17): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_bucket']


9 numeric features: ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_active_services', 'has_internet', 'avg_monthly_spend', 'charge_per_service', 'is_month_to_month', 'contract_risk_score']
17 categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_bucket']


In [6]:
X = df_feat.drop(columns=['Churn'])
X_transformed = preprocessor.fit_transform(X)
print('Transformed feature matrix shape:', X_transformed.shape)
print('One-hot encoding expands the categorical columns into many binary indicator columns.')

Transformed feature matrix shape: (7021, 56)
One-hot encoding expands the categorical columns into many binary indicator columns.


The transformed, model-ready matrix is what flows into [`src/models/train_model.py`](../src/models/train_model.py) — see **`03_modeling.ipynb`** for hyperparameter tuning and evaluation.